In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.stats import linregress
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

%matplotlib inline
np.random.seed(42)

## 1. Carregar dados

Mesma fonte usada no `02_modelo_regimes_hmm.ipynb`.

In [2]:
df_cesi = pd.read_csv('data/processed/cesi_limpo.csv', index_col=0, parse_dates=True)

dados_spy = yf.download("SPY", start="2004-01-01")
preco_spy = dados_spy['Close']
if isinstance(preco_spy, pd.DataFrame):
    preco_spy = preco_spy.iloc[:, 0]

retornos_spy = np.log(preco_spy / preco_spy.shift(1))

dados_modelo = pd.concat([retornos_spy, df_cesi], axis=1).dropna()
dados_modelo.columns = ['retorno'] + list(df_cesi.columns)
dados_modelo.head()

[*********************100%***********************]  1 of 1 completed


,retorno,cesi_global,cesi_g10,cesi_united_states,cesi_eurozone,cesi_asia_pacific,cesi_latin_america,cesi_emerging_markets,cesi_bric
Date,,,,,,,,,
2004-01-05,0.010820,40.5,55.5,71.6,53.6,36.3,-36.1,18.0,12.22
2004-01-06,0.000978,38.5,52.7,65.0,54.7,34.8,-35.1,17.1,12.36
2004-01-07,0.003370,38.1,50.7,62.3,52.1,37.7,-34.0,19.1,12.86
2004-01-08,0.003977,35.1,50.1,61.4,51.7,36.9,-39.9,12.5,9.10
2004-01-09,-0.008770,32.2,45.0,47.7,52.6,36.3,-35.8,13.1,8.81


In [3]:
WINDOW = 14
WINDOW_LONGO = WINDOW * 3
FEATURES = ['volatilidade', 'trend_r2', 'trend_r2_longo', 'autocorr', 'cesi_nivel', 'cesi_variacao']
COLUNA_CESI = 'cesi_global'  


def trend_r2(serie_precos):
    if serie_precos.isna().any():
        return np.nan
    x = np.arange(len(serie_precos))
    slope, _, r_value, _, _ = linregress(x, serie_precos.values)
    return (r_value ** 2) * np.sign(slope)


def autocorr_lag1(serie):
    if serie.isna().any() or serie.std() == 0:
        return np.nan
    return serie.autocorr(lag=1)


def construir_features(df_base, preco_serie, coluna_cesi=COLUNA_CESI):
    df_feat = df_base.copy()

    df_feat['volatilidade'] = df_feat['retorno'].rolling(WINDOW).std()
    df_feat['retorno_acumulado'] = df_feat['retorno'].rolling(WINDOW).sum()

    preco_alinhado = preco_serie.reindex(df_feat.index)
    df_feat['trend_r2'] = preco_alinhado.rolling(WINDOW).apply(trend_r2, raw=False)
    df_feat['trend_r2_longo'] = preco_alinhado.rolling(WINDOW_LONGO).apply(trend_r2, raw=False)
    df_feat['autocorr'] = df_feat['retorno'].rolling(WINDOW).apply(autocorr_lag1, raw=False)

    df_feat['cesi_nivel'] = df_feat[coluna_cesi]
    df_feat['cesi_variacao'] = df_feat[coluna_cesi].diff(WINDOW)

    return df_feat.dropna()


df_feat = construir_features(dados_modelo, preco_spy)
df_feat[FEATURES].describe()

,volatilidade,trend_r2,trend_r2_longo,autocorr,cesi_nivel,cesi_variacao
count,5628.000000,5628.000000,5628.000000,5628.000000,5628.000000,5628.000000
mean,0.009621,0.187309,0.270708,-0.101589,6.529279,0.047495
std,0.007009,0.512296,0.502957,0.240323,27.573971,12.064693
min,0.001546,-0.956526,-0.939459,-0.803432,-99.500000,-61.500000
25%,0.005577,-0.147685,-0.056004,-0.267418,-9.900000,-6.700000
50%,0.007789,0.182841,0.314798,-0.102177,5.850000,0.100000
75%,0.011308,0.661003,0.748633,0.066061,22.600000,6.925000
max,0.068276,0.975281,0.962510,0.728206,115.900000,73.800000


In [ ]:
TREINO_INICIAL = 756     
REFIT_A_CADA = 63        
N_ESTADOS = 3
FEATURES_NOMEACAO = ['trend_r2', 'trend_r2_longo']  


def nomear_estados(df_janela, estados_previstos, features_nomeacao):
    tmp = df_janela[features_nomeacao].copy()
    tmp['estado'] = estados_previstos
    resumo = tmp.groupby('estado')[features_nomeacao].mean()
    resumo['trend_medio'] = resumo[features_nomeacao].mean(axis=1)

    estado_alta = resumo['trend_medio'].idxmax()
    estado_baixa = resumo['trend_medio'].idxmin()
    estado_lateral = [e for e in resumo.index if e not in (estado_alta, estado_baixa)][0]

    return {estado_alta: 'Alta', estado_baixa: 'Baixa', estado_lateral: 'Lateral'}


def hmm_walk_forward(df_feat, features, treino_inicial=TREINO_INICIAL,
                      refit_a_cada=REFIT_A_CADA, n_estados=N_ESTADOS):
    n = len(df_feat)
    regimes = pd.Series(index=df_feat.index, dtype=object)

    inicio_teste = treino_inicial
    while inicio_teste < n:
        fim_treino = inicio_teste
        fim_teste = min(inicio_teste + refit_a_cada, n)

        janela_treino = df_feat.iloc[:fim_treino]
        janela_teste = df_feat.iloc[inicio_teste:fim_teste]

        scaler = StandardScaler()
        X_treino = scaler.fit_transform(janela_treino[features].values)
        X_teste = scaler.transform(janela_teste[features].values)

        modelo = GaussianHMM(n_components=n_estados, covariance_type='full',
                              n_iter=1000, random_state=42)
        modelo.fit(X_treino)

        estados_treino = modelo.predict(X_treino)
        mapa_nomes = nomear_estados(janela_treino, estados_treino, FEATURES_NOMEACAO)

        estados_teste = modelo.predict(X_teste)
        regimes.iloc[inicio_teste:fim_teste] = [mapa_nomes[e] for e in estados_teste]

        inicio_teste = fim_teste

    return regimes


df_feat['regime_wf'] = hmm_walk_forward(df_feat, FEATURES)
df_feat = df_feat.dropna(subset=['regime_wf'])
df_feat['regime_wf'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
cores = {'Lateral': 'gray', 'Alta': 'green', 'Baixa': 'red'}
preco_plot = preco_spy.reindex(df_feat.index)

for regime, cor in cores.items():
    mask = df_feat['regime_wf'] == regime
    ax.scatter(df_feat.index[mask], preco_plot[mask], c=cor, s=4, label=regime)

ax.set_title('SPY — Regimes (walk-forward, preço + CESI)')
ax.set_ylabel('Preço de fechamento')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
df_feat.groupby('regime_wf')[['cesi_nivel', 'cesi_variacao', 'trend_r2', 'volatilidade']].mean().round(2)

In [ ]:
MIN_DIAS_CONFIRMACAO = 5


def debounce_regime(regime_bruto, min_dias=MIN_DIAS_CONFIRMACAO):
    """Só confirma a troca de regime após `min_dias` observações consecutivas iguais."""
    valores = regime_bruto.values
    confirmado = np.empty(len(valores), dtype=object)

    regime_atual = valores[0]
    candidato = valores[0]
    contagem = 1

    for i, v in enumerate(valores):
        if v == candidato:
            contagem += 1
        else:
            candidato = v
            contagem = 1

        if contagem >= min_dias:
            regime_atual = candidato

        confirmado[i] = regime_atual

    return pd.Series(confirmado, index=regime_bruto.index)


df_feat['regime_filtrado'] = debounce_regime(df_feat['regime_wf'])

trocas_bruto = (df_feat['regime_wf'] != df_feat['regime_wf'].shift(1)).sum()
trocas_filtrado = (df_feat['regime_filtrado'] != df_feat['regime_filtrado'].shift(1)).sum()
print(f"Trocas de regime — bruto: {trocas_bruto} | filtrado: {trocas_filtrado} "
      f"({trocas_filtrado / trocas_bruto:.0%} do original)")

In [ ]:
MAPA_LS = {'Alta': 1, 'Lateral': 0, 'Baixa': -1}
MAPA_LONG = {'Alta': 1, 'Lateral': 0, 'Baixa': 0}

variantes_sinal = {
    'bruto_ls': df_feat['regime_wf'].map(MAPA_LS),
    'bruto_long': df_feat['regime_wf'].map(MAPA_LONG),
    'filtrado_ls': df_feat['regime_filtrado'].map(MAPA_LS),
    'filtrado_long': df_feat['regime_filtrado'].map(MAPA_LONG),
}

# lag de 1 dia em todas as variantes: só sabemos o regime de t após o fechamento de t
posicoes = {
    nome: sinal.shift(1).fillna(0)
    for nome, sinal in variantes_sinal.items()
}

pd.DataFrame({nome: pos.value_counts() for nome, pos in posicoes.items()}).fillna(0).astype(int)

## 5. Backtest (todas as variantes)

Mesma lógica de custo de transação de antes, agora encapsulada numa função e aplicada às 4
variantes.

In [ ]:
CUSTO_TRANSACAO = 0.0005  


def rodar_backtest(posicao, retorno, custo_transacao=CUSTO_TRANSACAO):
    troca = posicao.diff().abs().fillna(0)
    custo = troca * custo_transacao
    retorno_estrategia = posicao * retorno - custo
    equity = np.exp(retorno_estrategia.cumsum())
    return retorno_estrategia, equity


resultados = {}
for nome, pos in posicoes.items():
    ret_estrat, equity = rodar_backtest(pos, df_feat['retorno'])
    resultados[nome] = {'retorno': ret_estrat, 'equity': equity}

df_feat['retorno_benchmark'] = df_feat['retorno']
df_feat['equity_benchmark'] = np.exp(df_feat['retorno_benchmark'].cumsum())

fig, ax = plt.subplots(figsize=(16, 6))
for nome, r in resultados.items():
    ax.plot(df_feat.index, r['equity'], label=nome, alpha=0.85)
ax.plot(df_feat.index, df_feat['equity_benchmark'], label='Buy & Hold SPY',
        color='black', linewidth=1.5, alpha=0.6)
ax.set_title('Curva de capital — todas as variantes (preço + CESI) vs. Buy & Hold')
ax.set_ylabel('Capital (base 1.0)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
CORTE_OOS = '2020-01-01'  

idx_in = df_feat.index < CORTE_OOS
idx_oos = ~idx_in

print(f"In-sample:      {df_feat.index[idx_in].min().date()} a {df_feat.index[idx_in].max().date()} "
      f"({idx_in.sum()} dias)")
print(f"Out-of-sample:  {df_feat.index[idx_oos].min().date()} a {df_feat.index[idx_oos].max().date()} "
      f"({idx_oos.sum()} dias)")

## 6. Métricas de risco e retorno — comparação das variantes

In [ ]:
def metricas(retornos, dias_por_ano=252):
    retornos = retornos.dropna()
    ret_medio = retornos.mean() * dias_por_ano
    vol = retornos.std() * np.sqrt(dias_por_ano)
    sharpe = ret_medio / vol if vol > 0 else np.nan

    equity = np.exp(retornos.cumsum())
    n_anos = len(retornos) / dias_por_ano
    cagr = equity.iloc[-1] ** (1 / n_anos) - 1 if n_anos > 0 else np.nan

    drawdown = equity / equity.cummax() - 1
    max_dd = drawdown.min()

    win_rate = (retornos > 0).mean()

    return pd.Series({
        'Retorno anualizado': ret_medio,
        'Volatilidade anualizada': vol,
        'Sharpe': sharpe,
        'CAGR': cagr,
        'Máximo Drawdown': max_dd,
        'Win rate (dias)': win_rate,
    })


tabela = {}
for nome, r in resultados.items():
    tabela[f'{nome} (completo)'] = metricas(r['retorno'])
    tabela[f'{nome} (OOS)'] = metricas(r['retorno'][idx_oos])

tabela['Buy & Hold (completo)'] = metricas(df_feat['retorno_benchmark'])
tabela['Buy & Hold (OOS)'] = metricas(df_feat['retorno_benchmark'][idx_oos])

tabela_metricas = pd.DataFrame(tabela).round(3)
tabela_metricas[[c for c in tabela_metricas.columns if 'completo' in c]]

In [ ]:
tabela_metricas[[c for c in tabela_metricas.columns if 'OOS' in c]]

## 7. Teste de robustez em subperíodos



In [ ]:
MELHOR_VARIANTE = 'filtrado_long' 

subperiodos = {
    'Crise 2008-2009': ('2008-01-01', '2009-12-31'),
    'Covid 2020': ('2020-01-01', '2020-12-31'),
    'Alta de juros 2022': ('2022-01-01', '2022-12-31'),
    'Recente (últimos 2 anos)': (df_feat.index.max() - pd.DateOffset(years=2), df_feat.index.max()),
}

ret_melhor = resultados[MELHOR_VARIANTE]['retorno']

resultado_robustez = {}
for nome, (ini, fim) in subperiodos.items():
    janela_estrat = ret_melhor.loc[ini:fim]
    janela_bench = df_feat['retorno_benchmark'].loc[ini:fim]
    if len(janela_estrat) < 30:
        continue
    resultado_robustez[f'{nome} ({MELHOR_VARIANTE})'] = metricas(janela_estrat)
    resultado_robustez[f'{nome} (Buy & Hold)'] = metricas(janela_bench)

pd.DataFrame(resultado_robustez).round(3)